In [1]:
from openai import OpenAI

# Инициализируем клиент, который теперь точно увидит запущенную Ollama
ollama_client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"  # Заглушка (ключ не нужен)
)

try:
    # Отправляем тестовый запрос к вашей модели
    response = ollama_client.chat.completions.create(
        model="qwen3.5:4b",
        messages=[
            {"role": "user", "content": "Привет, Qwen! Подтверди, что соединение установлено успешно."}
        ]
    )
    print("--- Ответ от модели ---")
    print(response.choices[0].message.content)

except Exception as e:
    print(f"Что-то пошло не так: {e}")
    print("Если пишет, что модель не найдена, проверьте её точное имя через 'ollama list' в терминале.")

--- Ответ от модели ---
Приветствую вас, пользователь. Соединение работает в полной мере: всё функционирует стабильно и соответствует требованиям для эффективной работы с моими возможностями как языковой модели Qwen3-7B-Instruct. 

Если возникнут вопросы или потребуется помощь — обращайтесь любым удобным способом! 😊


In [2]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [5]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)
    
len(documents)

Q1 Answer: 72


In [19]:
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)

question = "How does the agentic loop keep calling the model until it stops?"

search_results = index.search(
    question,
    num_results=5
)

search_results[0]['filename']

'01-agentic-rag/lessons/14-agentic-loop.md'

In [17]:
from dataclasses import dataclass

INSTRUCTIONS = '''
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
'''

PROMPT_TEMPLATE = '''
QUESTION: {question}

CONTEXT:
{context}
'''.strip()


@dataclass
class RAGResponse:
    answer: str
    prompt_tokens: int


class LessonRAG:

    def __init__(
        self,
        index,
        llm_client,
        instructions=INSTRUCTIONS,
        prompt_template=PROMPT_TEMPLATE,
        model='qwen3.5:4b'
    ):
        self.index = index
        self.llm_client = llm_client
        self.instructions = instructions
        self.prompt_template = prompt_template
        self.model = model

    def search(self, query, num_results=5):
        return self.index.search(
            query,
            num_results=num_results
        )

    def build_context(self, search_results):
        lines = []

        for doc in search_results:
            lines.append(f"Filename: {doc['filename']}")
            lines.append(f"Content: {doc['content']}")
            lines.append('')

        return '\n'.join(lines).strip()

    def build_prompt(self, query, search_results):
        context = self.build_context(search_results)
        return self.prompt_template.format(
            question=query, context=context
        )

    def llm(self, prompt):
        input_messages = [
            {'role': 'developer', 'content': self.instructions},
            {'role': 'user', 'content': prompt}
        ]

        response = self.llm_client.responses.create(
            model=self.model,
            input=input_messages
        )
        return response

    def rag(self, query):
        search_results = self.search(query)
        prompt = self.build_prompt(query, search_results)
        
        response = self.llm(prompt)
        
        return RAGResponse(
            answer=response.output_text,
            prompt_tokens=response.usage.input_tokens
        )
    
rag_assistant = LessonRAG(
    index=index,
    llm_client=ollama_client,
    model='qwen3.5:4b'
)

query = "How does the agentic loop keep calling the model until it stops?"
result = rag_assistant.rag(query)
result.prompt_tokens

4095

In [18]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
len(chunks)

295

In [20]:
chunk_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)
chunk_index.fit(chunks)
chunk_rag_assistant = LessonRAG(
    index=chunk_index,
    llm_client=ollama_client,
    model='qwen3.5:4b'
)
result_chunk = chunk_rag_assistant.rag(query)
result_chunk.prompt_tokens

2430

In [28]:
instructions = "You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering."

In [42]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

ModuleNotFoundError: No module named 'toyaikit.llm.clients'; 'toyaikit.llm' is not a package

In [46]:
search_counter = 0

def search(query: str) -> dict[str, str]:
    global search_counter
    search_counter += 1
    
    results = chunk_index.search(
        query,
        num_results=5,
    )
    
    return str(results)

agent_tools = Tools()
agent_tools.add_tool(search)
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'No description provided.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [47]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)


client = OpenAIClient(
    model='qwen3.5:4b',
    client=ollama_client
)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=client
)

search_counter = 0

In [49]:
result = runner.loop(
    prompt='How does the agentic loop work, and how is it different from plain RAG?',
    callback=callback,
)

search_counter

-> Response received


-> Response received


Component,Description,Typical Use Case
Workflow,Linear pipeline: Search → Build Prompt → Generate Answer,Static questions where the answer can be found in documents without needing multiple reasoning steps or iterative refinement
Iteration Count,Single LLM call after retrieval (occasionally with 1-2 calls total),"Fixed, deterministic information queries like ""What's my policy on X?"""
Decision Logic,"Static: always search first, then generate final answer",No dynamic decision-making; logic is predefined and not affected by intermediate results
Component,Description,Typical Use Case
Workflow,"Iterative cycle: Question → Decision/Action (Tools, Search, Reasoning) → Update Context → Repeat until goal achieved","Open-ended tasks requiring multi-step decision making or tool usage, such as ""Research data engineering trends and create a report"""
Iteration Count,Multiple LLM calls; model decides when to continue based on state/goal satisfaction,"Tasks needing multiple iterations, e.g., error recovery (misspelling correction in search), iterative refinement of answers through several steps"
Decision Logic,"Dynamic: Model evaluates results and determines if further action is needed, or it has tool use capabilities. For example, model sees ""search returned poor results"" then retries with different query term like ""Ollama"". When conditions are met (e.g., no more tool calls), the agent stops and returns final answer",Decision-making based on dynamic information; adapts to unexpected situations without human intervention
Aspect,RAG System,Agentic Loop
State,Single-pass; uses retrieved context to generate one output,Maintains conversation history (all_messages) across iterations and decisions based on prior results
Tool Use,Limited or none (can be part of retrieval step but not agent-controlled),"Built-in capability for tool invocation, execution results are fed back into next iteration"


2